# EXPERIMENT 3 (Table 1)

Reproducing Table 1: axiom satisfaction of three architectures (MLP, CNN, WEC) trained unsupervised on axiom losses, plus 16 known voting rules for comparison. IC sampling, 55 voters, 5 alternatives.

Common hyperparameters (paper §5.1, §6.3):
* 55 voters, 5 alternatives, IC sampling
* 15,000 gradient steps, batch size 200, AdamW
* Cosine annealing with warm restarts (T_0 = 100)
* `sample_size_applicable = 400` for axiom checking

Architecture-specific:
* MLP: 4 hidden layers à 128. Optimizes NW, Anonymity, Condorcet, Pareto, Independence.
* CNN: kernel1 = (5,1), kernel2 = (1,5), 64 channels, 3 linear layers à 128. Same axioms as MLP.
* WEC: pretrained Word2Vec (corpus 100k, dim 200, window 7, skip-gram) + averaging + 3 linear layers à 128. Optimizes NW, Condorcet, Pareto only (anonymous by design).

Decoding variants reported in Table 1: plain (`p`), neutrality-averaged (`n`), neutrality-and-anonymity-averaged (`na`). MLP/CNN report all three; WEC reports `p` and `n` only (anonymous by design).

Paper run times (Fig. 16): MLP ≈ 5h 29min, CNN ≈ 6h 08min, WEC ≈ 2h 05min on a CPU laptop.

In [1]:
import exp3
import utils
import plot_and_visual

MAX_NUM_VOTERS = 55
MAX_NUM_ALTERNATIVES = 5
ELECTION_SAMPLING = {'probmodel': 'IC'}
NUM_GRADIENT_STEPS = 15000
LOSS_REPORT_INTERVALS = 100
BATCH_SIZE = 200
LEARNING_SCHEDULER = 100
EVAL_DATASET_SIZE = 500
SAMPLE_SIZE_APPLICABLE = 400
SAMPLE_SIZE_MAXIMAL = int(1e6)
DISTANCE = 'KLD'

AXIOMS_CHECK_MODEL = ['Anonymity', 'Neutrality', 'Condorcet', 'Pareto', 'Independence']

AXIOM_OPT_MLP_CNN = {
    'No_winner':   {'weight': 10, 'period': 'always'},
    'All_winners': None,
    'Inadmissible': None,
    'Resoluteness': None,
    'Parity':      None,
    'Anonymity':   {'weight': 1, 'period': 'always', 'sample': 50},
    'Neutrality':  None,
    'Condorcet1':  {'weight': 2, 'period': 'always'},
    'Condorcet2':  None,
    'Pareto1':     None,
    'Pareto2':     {'weight': 1, 'period': 'always'},
    'Independence':{'weight': 1, 'period': 'always', 'sample': 4},
}

AXIOM_OPT_WEC = {
    'No_winner':   {'weight': 10, 'period': 'always'},
    'All_winners': None,
    'Inadmissible': None,
    'Resoluteness': None,
    'Parity':      None,
    'Anonymity':   None,
    'Neutrality':  None,
    'Condorcet1':  {'weight': 2, 'period': 'always'},
    'Condorcet2':  None,
    'Pareto1':     None,
    'Pareto2':     {'weight': 1, 'period': 'always'},
    'Independence':None,
}

In [ ]:
print('Training MLP (NW, A, C, P, I)')
location_mlp = exp3.experiment3(
    architecture = 'MLP',
    max_num_voters = MAX_NUM_VOTERS,
    max_num_alternatives = MAX_NUM_ALTERNATIVES,
    election_sampling = ELECTION_SAMPLING,
    num_gradient_steps = NUM_GRADIENT_STEPS,
    report_intervals = NUM_GRADIENT_STEPS,
    eval_dataset_size = EVAL_DATASET_SIZE,
    model_to_rule = {
        'plain': True,
        'neut-averaged': None,
        'neut-anon-averaged': [12, 10],
    },
    sample_size_applicable = SAMPLE_SIZE_APPLICABLE,
    sample_size_maximal = SAMPLE_SIZE_MAXIMAL,
    architecture_parameters = None,
    axioms_check_model = AXIOMS_CHECK_MODEL,
    axioms_check_rule = [],
    axiom_opt = AXIOM_OPT_MLP_CNN,
    comp_rules_axioms = [],
    comp_rules_similarity = [],
    distance = DISTANCE,
    batch_size = BATCH_SIZE,
    learning_scheduler = LEARNING_SCHEDULER,
)
print(f'MLP results: {location_mlp}')

In [ ]:
print('Training CNN (NW, A, C, P, I)')
location_cnn = exp3.experiment3(
    architecture = 'CNN',
    max_num_voters = MAX_NUM_VOTERS,
    max_num_alternatives = MAX_NUM_ALTERNATIVES,
    election_sampling = ELECTION_SAMPLING,
    num_gradient_steps = NUM_GRADIENT_STEPS,
    report_intervals = NUM_GRADIENT_STEPS,
    loss_report_intervals=LOSS_REPORT_INTERVALS,
    eval_dataset_size = EVAL_DATASET_SIZE,
    model_to_rule = {
        'plain': True,
        'neut-averaged': None,
        'neut-anon-averaged': [12, 10],
    },
    sample_size_applicable = SAMPLE_SIZE_APPLICABLE,
    sample_size_maximal = SAMPLE_SIZE_MAXIMAL,
    architecture_parameters = {
        'kernel1': [5, 1],
        'kernel2': [1, 5],
        'channels': 64,
    },
    axioms_check_model = AXIOMS_CHECK_MODEL,
    axioms_check_rule = [],
    axiom_opt = AXIOM_OPT_MLP_CNN,
    comp_rules_axioms = [],
    comp_rules_similarity = [],
    distance = DISTANCE,
    batch_size = BATCH_SIZE,
    learning_scheduler = LEARNING_SCHEDULER,
)
print(f'CNN results: {location_cnn}')

In [2]:
print('Training WEC (NW, C, P)')
location_wec = exp3.experiment3(
    architecture = 'WEC',
    max_num_voters = MAX_NUM_VOTERS,
    max_num_alternatives = MAX_NUM_ALTERNATIVES,
    election_sampling = ELECTION_SAMPLING,
    num_gradient_steps = NUM_GRADIENT_STEPS,
    # report_intervals = NUM_GRADIENT_STEPS // 5, # these are more like eval_intervals
    report_intervals = NUM_GRADIENT_STEPS, # these are more like eval_intervals
    loss_report_intervals=LOSS_REPORT_INTERVALS,
    eval_dataset_size = EVAL_DATASET_SIZE,
    model_to_rule = {
        'plain': True,
        'neut-averaged': None,
        'neut-anon-averaged': False,
    },
    sample_size_applicable = SAMPLE_SIZE_APPLICABLE,
    sample_size_maximal = SAMPLE_SIZE_MAXIMAL,
    architecture_parameters = {
        'we_corpus_size': int(1e5),
        'we_size': 200,
        'we_window': 7,
        'we_algorithm': 1,
    },
    axioms_check_model = AXIOMS_CHECK_MODEL,
    axioms_check_rule = [],
    axiom_opt = AXIOM_OPT_WEC,
    comp_rules_axioms = [],
    comp_rules_similarity = [],
    distance = DISTANCE,
    batch_size = BATCH_SIZE,
    learning_scheduler = LEARNING_SCHEDULER,
    save_model=True,
)
print(f'WEC results: {location_wec}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/felixwu/.netrc.


Training WEC (NW, C, P)
Saving location: ./results/exp3/WEC/exp3_2026-05-10_19-05-20_IC


wandb: Currently logged in as: felix_wu (felix_wu-technische-universit-t-m-nchen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Now pretraining word embeddings
First generate profiles and turn them into corpus
Now train word embeddings
Done pretraining word embeddings.
Save the word embeddings.
Now generate dev and test profiles
Now starting to train


100%|█████████▉| 14999/15000 [29:48<00:00,  8.21it/s]

...

100%|██████████| 15000/15000 [45:17<00:00,  5.52it/s] 

    Independence 41.25%
Admissability on test profiles (plain):
    no_winner_at_all 0.004
    no_admissible_winner 0.004
    all_admissible_winner 0.222
    some_inadmissible_winner 0.0


Admissability on test profiles (neutrality-averaged):
    no_winner_at_all 0.012
    no_admissible_winner 0.012
    all_admissible_winner 0.228
    some_inadmissible_winner 0.0
Axiom satisfaction (plain):
    Anonymity 100.0%
    Neutrality 74.75%
    Condorcet 95.5%
    Pareto 100.0%
    Independence 36.25%
Axiom satisfaction (neutrality-averaged):
    Anonymity 100.0%
    Neutrality 100.0%
    Condorcet 98.25%
    Pareto 100.0%
    Independence 43.75%
Axiom satisfaction of the rules:
Runtime (in min): 65


loss/loss_alwi,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/loss_anon,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/loss_con1,▇█▇▅▄▆▆▄▅▆▄▄▁▅▅▄▄▂▂▄▂▃▃▂▃▄▃▃▃▄▅▂▁▂▁▃▁▂▁▃
loss/loss_con2,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/loss_inad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/loss_inde,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/loss_neut,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/loss_nowi,▁▁▁▁▁▂▅▁▂█▂▅▁▂▃▂▂▂▂▁▁▁▃▁▅▁▁▁▁▁▁▁▁▁▂▂▁▂▁▂
loss/loss_par1,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/loss_par2,▇█▆▂▇▂▄▄▁▄▁▄▃▄▆▂▄▅▃▄▂▃▃▂▁▄▃▃▃▂▁▃▂▃▁▂▁▂▃▂
+21,...


WEC results: ./results/exp3/WEC/exp3_2026-05-10_19-05-20_IC


In [ ]:
print('Computing axiom satisfaction of the 16 comparison rules')
location_rules = exp3.experiment3(
    architecture = 'MLP',
    max_num_voters = MAX_NUM_VOTERS,
    max_num_alternatives = MAX_NUM_ALTERNATIVES,
    election_sampling = ELECTION_SAMPLING,
    num_gradient_steps = 0,
    report_intervals = 1,
    eval_dataset_size = EVAL_DATASET_SIZE,
    model_to_rule = {
        'plain': False,
        'neut-averaged': False,
        'neut-anon-averaged': False,
    },
    sample_size_applicable = SAMPLE_SIZE_APPLICABLE,
    sample_size_maximal = SAMPLE_SIZE_MAXIMAL,
    architecture_parameters = None,
    axioms_check_model = [],
    axioms_check_rule = list(utils.dict_axioms_rules.keys()),
    axiom_opt = {
        'No_winner': None,
        'All_winners': None,
        'Inadmissible': None,
        'Resoluteness': None,
        'Parity': None,
        'Anonymity': None,
        'Neutrality': None,
        'Condorcet1': None,
        'Condorcet2': None,
        'Pareto1': None,
        'Pareto2': None,
        'Independence': None,
    },
    comp_rules_axioms = list(utils.dict_rules_all_fast.keys()),
    comp_rules_similarity = [],
)
print(f'Rule baseline results: {location_rules}')

In [ ]:
plot_and_visual.exp3_table(
    file_rules = 'results/exp3/MLP/exp3_2026-05-10_18-24-53_IC',
    file_MLP   = 'results/exp3/MLP/exp3_2026-05-05_11-04-57_IC',
    file_CNN   = 'results/exp3/CNN/exp3_2026-05-10_18-53-58_IC',
    file_WEC   = 'results/exp3/WEC/exp3_2026-05-10_19-05-20_IC',
)

KeyError: 'axiom_satisfaction'